In [1]:
import pandas as pd
import asyncio
import aiohttp
import os

In [2]:
df = pd.read_excel("img_metadata.xlsx")
df.head()

,filepath,filename,diag,fst,partition,md5hash,fitzpatrick,label,nine_partition_label,three_partition_label,qc,url,url_alphanum,orig_img_name,new_img_name
0,ne-de_361/ne-de_f2_61_3a96728f.jpg,ne-de_f2_61_3a96728f,ne-de,2,train,3a96728fb161fc25b3597d3996dc4e4a,2,neutrophilic dermatoses,inflammatory,non-neoplastic,NaN,https://www.dermaamin.com/site/images/clinical...,httpwwwdermaamincomsiteimagesclinicalpicppyode...,3a96728fb161fc25b3597d3996dc4e4a.jpg,ne-de_f2_61_3a96728f.jpg
1,lu-er_410/lu-er_f1_89_07be6be3.jpg,lu-er_f1_89_07be6be3,lu-er,1,train,07be6be3cbebd04ff9bfcab50b75378b,1,lupus erythematosus,inflammatory,non-neoplastic,NaN,https://www.dermaamin.com/site/images/clinical...,httpwwwdermaamincomsiteimagesclinicalpicssubac...,07be6be3cbebd04ff9bfcab50b75378b.jpg,lu-er_f1_89_07be6be3.jpg
2,me_261/me_f0_194_72bcd875.jpg,me_f0_194_72bcd875,me,0,train,72bcd875e7a573b4139883059e69328d,-1,melanoma,malignant melanoma,malignant,NaN,http://atlasdermatologico.com.br/img?imageId=4264,httpwwwatlasdermatologicocombrimgimageId4264.jpg,72bcd875e7a573b4139883059e69328d.jpg,me_f0_194_72bcd875.jpg
3,dr-er_200/dr-er_f4_36_dfa97968.jpg,dr-er_f4_36_dfa97968,dr-er,4,train,dfa9796851f899e99961603c951742fa,4,drug eruption,inflammatory,non-neoplastic,NaN,https://www.dermaamin.com/site/images/clinical...,httpwwwdermaamincomsiteimagesclinicalpicddruge...,dfa9796851f899e99961603c951742fa.jpg,dr-er_f4_36_dfa97968.jpg
4,ju-xa_149/ju-xa_f2_118_be506d4a.jpg,ju-xa_f2_118_be506d4a,ju-xa,2,train,be506d4a799385099ac85daa695d6df3,2,juvenile xanthogranuloma,inflammatory,non-neoplastic,NaN,http://atlasdermatologico.com.br/img?imageId=8098,httpwwwatlasdermatologicocombrimgimageId8098.jpg,be506d4a799385099ac85daa695d6df3.jpg,ju-xa_f2_118_be506d4a.jpg


In [3]:
keep = [
    "url", # Image
    "fst", # Skin tone
    "partition", # train / test / valid
    "diag", # Short diagnostic
    "label", # Diagnostic
    "nine_partition_label", # Disease category
    "three_partition_label" # #Superclass
]
df = df[keep].copy()
df = df.dropna(subset=["url"])
df.head()

,url,fst,partition,diag,label,nine_partition_label,three_partition_label
0,https://www.dermaamin.com/site/images/clinical...,2,train,ne-de,neutrophilic dermatoses,inflammatory,non-neoplastic
1,https://www.dermaamin.com/site/images/clinical...,1,train,lu-er,lupus erythematosus,inflammatory,non-neoplastic
2,http://atlasdermatologico.com.br/img?imageId=4264,0,train,me,melanoma,malignant melanoma,malignant
3,https://www.dermaamin.com/site/images/clinical...,4,train,dr-er,drug eruption,inflammatory,non-neoplastic
4,http://atlasdermatologico.com.br/img?imageId=8098,2,train,ju-xa,juvenile xanthogranuloma,inflammatory,non-neoplastic


In [4]:
for split in df["partition"].value_counts().index:
    for label in df["nine_partition_label"].value_counts().index:
        os.makedirs(f"dataset/{split}/{label}", exist_ok=True)

In [5]:
import requests
from PIL import Image
from io import BytesIO

In [6]:
def download(url, save_path):
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()

        img = Image.open(BytesIO(r.content)).convert("RGB")
        img.save(save_path)
        return True
    except Exception as e:
        print(f"Failed: {url} | Error: {e}")
        return False

In [7]:
def save(df):
    for idx, row in df.iterrows():
        url = row["url"]
        save_path = f"dataset/{row["partition"]}/{row["nine_partition_label"]}/{idx}.jpg"
        download(url, save_path)

In [8]:
save(df)

Failed: http://atlasdermatologico.com.br/img?imageId=2860 | Error: 404 Client Error:  for url: https://atlasdermatologico.com.br/img?imageId=2860
Failed: http://atlasdermatologico.com.br/img?imageId=6724 | Error: 404 Client Error:  for url: https://atlasdermatologico.com.br/img?imageId=6724
Failed: http://atlasdermatologico.com.br/img?imageId=2862 | Error: 404 Client Error:  for url: https://atlasdermatologico.com.br/img?imageId=2862
Failed: http://atlasdermatologico.com.br/img?imageId=4505 | Error: 404 Client Error:  for url: https://atlasdermatologico.com.br/img?imageId=4505
Failed: https://www.dermaamin.com/site/images/clinical-pic/d/disseminated_porokeratosis/disseminated_porokeratosis5.jpg | Error: HTTPSConnectionPool(host='www.dermaamin.com', port=443): Read timed out.
Failed: http://atlasdermatologico.com.br/img?imageId=2859 | Error: 404 Client Error:  for url: https://atlasdermatologico.com.br/img?imageId=2859
Failed: http://atlasdermatologico.com.br/img?imageId=8363 | Error: 4

In [4]:
import torch
import torchvision
import torchvision.transforms as transforms

train_tfms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor()
])

train_dataset = torchvision.datasets.ImageFolder("dataset/train", transform = train_tfms)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=False)

In [7]:
def get_mean_std(loader):
    mean = 0
    std = 0
    size = len(loader.dataset)
    for images, labels in loader:
        batch_size = images.size(0)
        images = images.view(batch_size, images.size(1), -1)
        mean += images.mean(2).sum(0)
        std += images.std(2).sum(0)

    mean /= size
    std /= size

    return mean, std

In [8]:
get_mean_std(train_loader)

(tensor([0.6505, 0.4933, 0.4403]), tensor([0.1574, 0.1441, 0.1420]))